# 13. Despliegue de Modelos de Machine Learning y Deep Learning

Este notebook cubre el proceso de despliegue de modelos usando FastAPI, Docker y buenas prácticas para APIs de inferencia.

## Objetivo
- Aprender a serializar y cargar modelos entrenados.
- Crear una API REST de inferencia con FastAPI.
- Conocer las bases de contenedorización con Docker.
- Aplicar buenas prácticas de despliegue y versionamiento.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado al menos los notebooks [01](./01_intro_machine_learning.ipynb) y [03](./03_modelos_clasicos_ml.ipynb).

- Conceptos de entrenamiento y evaluación de modelos.

> ⚠️ **Dependencias adicionales:** `pip install fastapi uvicorn nest_asyncio`

## 1. Introducción teórica

El despliegue es el paso final del ciclo de vida de ML: llevar un modelo entrenado a producción para que pueda ser usado por aplicaciones reales.

**Etapas del despliegue:**
1. Serialización del modelo (guardar/cargar).
2. Creación de una API de inferencia.
3. Contenedorización (Docker).
4. Despliegue en la nube o servidor.

| Herramienta | Uso |
|-------------|-----|
| **joblib / pickle** | Serialización de modelos scikit-learn |
| **model.save()** | Serialización de modelos Keras |
| **FastAPI** | API REST de inferencia |
| **Docker** | Contenedorización para portabilidad |
| **MLflow / DVC** | Versionamiento de modelos y experimentos |

## 2. Entrenar y guardar un modelo

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

# Entrenar modelo de ejemplo
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print(f'Accuracy en test: {accuracy_score(y_test, clf.predict(X_test)):.2f}')

# Guardar modelo
joblib.dump(clf, 'modelo_iris.joblib')
print('Modelo guardado como modelo_iris.joblib')

Accuracy en test: 1.00
Modelo guardado como modelo_iris.joblib


## 3. Cargar el modelo guardado

In [2]:
# Cargar modelo guardado
del clf
modelo = joblib.load('modelo_iris.joblib')

# Probar predicción
sample = X[0:1]
pred = modelo.predict(sample)
prob = modelo.predict_proba(sample)
print(f'Predicción: {data.target_names[pred[0]]} ({pred[0]})')
print(f'Probabilidades: {prob[0].round(3)}')

Predicción: setosa (0)
Probabilidades: [1. 0. 0.]


## 4. Crear y lanzar una API REST desde el notebook

Usaremos FastAPI y `nest_asyncio` para lanzar la API dentro del notebook.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
import nest_asyncio
import uvicorn
from threading import Thread

# Cargar modelo
modelo = joblib.load('modelo_iris.joblib')

app = FastAPI(title='Iris Classifier API', version='1.0')

class InputData(BaseModel):
    features: list

class PredictionResponse(BaseModel):
    prediction: int
    class_name: str
    probabilities: list

@app.post('/predict', response_model=PredictionResponse)
def predict(data: InputData):
    X = np.array(data.features).reshape(1, -1)
    pred = modelo.predict(X)
    proba = modelo.predict_proba(X)[0].tolist()
    class_names = ['setosa', 'versicolor', 'virginica']
    return PredictionResponse(
        prediction=int(pred[0]),
        class_name=class_names[int(pred[0])],
        probabilities=proba
    )

@app.get('/health')
def health():
    return {'status': 'ok'}

# Permitir reinicio de bucle de eventos en Jupyter
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host='0.0.0.0', port=8000)

# Lanzar API en un hilo
api_thread = Thread(target=run_api, daemon=True)
api_thread.start()
print('API lanzada en http://localhost:8000')
print('Documentación automática en http://localhost:8000/docs')

API lanzada en http://localhost:8000
Documentación automática en http://localhost:8000/docs


INFO:     Started server process [6692]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


## 5. Probar la API localmente

In [5]:
import requests
import time

# Esperar a que la API esté lista
time.sleep(2)

# Health check
try:
    health = requests.get('http://127.0.0.1:8000/health')
    print('Health check:', health.json())
except:
    print('La API aún no está lista, intenta más tarde.')

# Predicción
url = 'http://127.0.0.1:8000/predict'
data = {"features": [5.1, 3.5, 1.4, 0.2]}
response = requests.post(url, json=data)
print('Respuesta de la API:', response.json())

INFO:     127.0.0.1:55357 - "GET /health HTTP/1.1" 200 OK
Health check: {'status': 'ok'}
INFO:     127.0.0.1:55358 - "POST /predict HTTP/1.1" 200 OK
Respuesta de la API: {'prediction': 0, 'class_name': 'setosa', 'probabilities': [1.0, 0.0, 0.0]}


## 6. Contenedorización con Docker

Para desplegar en cualquier servidor, creamos un contenedor Docker. A continuación, los archivos necesarios:

### `Dockerfile`

```dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY modelo_iris.joblib .
COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

### `requirements.txt`

```
fastapi
uvicorn
scikit-learn
numpy
joblib
```

### `docker-compose.yml`

```yaml
version: '3.8'
services:
  api:
    build: .
    ports:
      - "8000:8000"
    restart: unless-stopped
```

### Comandos para desplegar:

```bash
# Construir imagen
docker build -t iris-api .

# Ejecutar contenedor
docker run -p 8000:8000 iris-api

# Con docker-compose
docker-compose up -d
```

## 7. Versionamiento de modelos

En producción, es esencial versionar los modelos:

| Herramienta | Ventaja | Cuándo usar |
|-------------|---------|-------------|
| **MLflow** | Tracking de experimentos, registro de modelos | Equipos medianos/grandes |
| **DVC** | Control de versiones de datos y modelos | Integración con Git |
| **Convención manual** | Nombres con timestamp y métricas | Proyectos pequeños |

**Ejemplo de versionamiento manual:**

In [10]:
print(type(data))
print(data.keys() if isinstance(data, dict) else "No es dict")

<class 'dict'>
dict_keys(['features'])


In [13]:
from datetime import datetime
import json

# Guardar modelo con versión
version = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = f'modelo_iris_v{version}.joblib'
joblib.dump(modelo, model_path)

# Recuperamos los nombres del dataset original (cargado al inicio)
# Si 'data' fue sobreescrito, usa los nombres estándar de Iris:
feature_names = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
target_names = ['setosa', 'versicolor', 'virginica']

metadata = {
    'version': version,
    'model_type': 'RandomForestClassifier',
    'accuracy': accuracy_score(y_test, modelo.predict(X_test)),
    'features': feature_names,
    'target_names': target_names
}

with open(f'modelo_iris_v{version}_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Modelo guardado: {model_path}')
print(f'Metadata: {json.dumps(metadata, indent=2)}')

Modelo guardado: modelo_iris_v20260331_152959.joblib
Metadata: {
  "version": "20260331_152959",
  "model_type": "RandomForestClassifier",
  "accuracy": 1.0,
  "features": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ],
  "target_names": [
    "setosa",
    "versicolor",
    "virginica"
  ]
}


## 8. Discusión y Conclusiones

- Serializar modelos con `joblib` es simple y eficiente para scikit-learn.
- FastAPI permite crear APIs de inferencia rápidamente con documentación automática.
- Docker garantiza portabilidad y consistencia del entorno.
- El versionamiento de modelos es crítico para reproducibilidad en producción.
- Siempre incluir validación de datos, health checks, y manejo de errores.

## 9. Ejercicios Propuestos

1. **Ejercicio 1:** Modifica la API para que devuelva también las probabilidades por clase en formato de diccionario con los nombres de las clases.

2. **Ejercicio 2:** Agrega validación de entrada (que `features` tenga exactamente 4 elementos y sean numéricos).

3. **Ejercicio 3:** Crea los archivos `Dockerfile`, `requirements.txt` y `app.py` y construye la imagen Docker.

4. **Ejercicio 4 (Avanzado):** Configura MLflow para trackear experimentos y registra el modelo con diferentes hiperparámetros.

## Resolución de Ejercicios Propuestos

A continuación, implementaremos la solución integral para los 4 ejercicios propuestos, llevando nuestro modelo desde un entorno de experimentación local hasta un servicio profesional.

Los módulos a desarrollar son:
1. **Mejora y Validación de la API (Ejercicios 1 y 2):** Se añadirá validación estricta de entrada con Pydantic (garantizando 4 features numéricas) y se enriquecerá la respuesta para entregar un diccionario de probabilidades mapeadas por clase.
2. **Contenedorización (Ejercicio 3):** Crearemos físicamente los archivos `app.py`, `requirements.txt` y `Dockerfile` necesarios para desplegar la API en un entorno aislado y reproducible.
3. **MLOps con MLflow (Ejercicio 4):** Implementaremos un pipeline avanzado de validación que registra parámetros, métricas y el modelo resultante utilizando MLflow.

### Ejercicios 1, 2 y 3.1: Construcción de `app.py`
En el siguiente bloque de código consolidamos la API REST usando FastAPI. 
- **Validación (Ej2):** Utilizamos sentencias de Pydantic (`field_validator` o validaciones manuales) para obligar a que `features` conste de exactamente 4 números.
- **Probabilidades (Ej1):** Construimos un diccionario mapeando los nombres *setosa, versicolor, virginica* con la probabilidad devuelta por el modelo.
Nota: Guardaremos el código directamente en el archivo `app.py`.

In [18]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, validator
import joblib
import numpy as np

# Cargar el modelo preentrenado
modelo = joblib.load('modelo_iris.joblib')

# Inicializar API
app = FastAPI(title='Iris Classifier API Pro', version='2.0')

# ==========================================
# EJERCICIO 2: Validación de la entrada
# ==========================================
class InputData(BaseModel):
    # Obligamos a que sea una lista de floats
    features: list[float]
    
    @validator('features')
    def validate_features_length(cls, v):
        if len(v) != 4:
            raise ValueError('¡Error! Se esperan exactamente 4 características numéricas.')
        return v

# ==========================================
# EJERCICIO 1: Diccionario de Probabilidades
# ==========================================
class PredictionResponse(BaseModel):
    prediction: int
    class_name: str
    probabilities: dict  # Ahora es un diccionario en lugar de una lista plana

@app.post('/predict', response_model=PredictionResponse)
def predict(data: InputData):
    try:
        # Preparar los datos
        X = np.array(data.features).reshape(1, -1)
        
        # Predecir
        pred = modelo.predict(X)
        proba = modelo.predict_proba(X)[0] # Probabilidades planas
        
        # Clases de la flor
        class_names = ['setosa', 'versicolor', 'virginica']
        
        # (EJERCICIO 1) Crear diccionario dinámico de probabilidades
        prob_dict = {class_names[i]: float(proba[i]) for i in range(len(class_names))}
        
        return PredictionResponse(
            prediction=int(pred[0]),
            class_name=class_names[int(pred[0])],
            probabilities=prob_dict
        )
    except Exception as e:
        # Manejo de cualquier otro error para que no caiga el servidor
        raise HTTPException(status_code=400, detail=str(e))

@app.get('/health')
def health():
    return {'status': 'ok', 'message': 'API funcionando correctamente'}

Writing app.py


### Ejercicio 3.2 y 3.3: `requirements.txt` y `Dockerfile`
A continuación, crearemos las instrucciones necesarias para que Docker pueda construir un contenedor liviano que aloje nuestra API.

In [19]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
numpy
joblib
pydantic

Writing requirements.txt


In [20]:
%%writefile Dockerfile
# Utilizar una imagen oficial y liviana de Python
FROM python:3.11-slim

# Definir el directorio de trabajo dentro del contenedor
WORKDIR /app

# Copiar el archivo de requerimientos e instalar dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar el código fuente de la API y el modelo preentrenado
COPY app.py .
COPY modelo_iris.joblib .

# Exponer el puerto en el que corre Uvicorn
EXPOSE 8000

# Comando para levantar la API al arrancar el contenedor
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


### Ejercicio 4: Tracking de Experimentos con MLflow
En este script, vamos a importar MLflow, entrenar un par de variaciones del modelo utilizando diferentes hiperparámetros (ej: la profundidad de un árbol) y dejaremos registro automático de su rendimiento (Accuracy) y del modelo generado.

In [21]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Preparar datos
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Nombre de nuestro experimento principal 
mlflow.set_experiment("Clasificador_Iris_rf")

# 2. Vamos a probar dos hiperparámetros diferentes
n_estimators_list = [10, 50]

for n_estimators in n_estimators_list:
    # Iniciar la "corrida" (run) dentro de MLflow
    with mlflow.start_run(run_name=f"RF_estimators_{n_estimators}"):
        
        # Crear y entrenar modelo
        clf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
        clf.fit(X_train, y_train)
        
        # Predecir y evaluar
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        # ---- MAGIA DE MLFLOW AQUÍ ----
        # Registramos el parámetro usado
        mlflow.log_param("n_estimators", n_estimators)
        
        # Registramos la métrica de éxito
        mlflow.log_metric("accuracy", acc)
        
        # Guardamos el modelo dentro de mlflow
        mlflow.sklearn.log_model(clf, f"modelo_rf_{n_estimators}")
        
        print(f"Ejecución con {n_estimators} estimadores finalizada. Accuracy: {acc:.4f}")

print("\n¡Tracking finalizado! Puedes abrir la consola de MLflow en tu terminal ejecutando: mlflow ui")

2026/03/31 16:00:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/31 16:00:45 INFO mlflow.store.db.utils: Updating database tables
2026/03/31 16:00:49 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Iris_rf' does not exist. Creating a new experiment.
2026/03/31 16:00:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 16:00:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ejecución con 10 estimadores finalizada. Accuracy: 1.0000


2026/03/31 16:01:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 16:01:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ejecución con 50 estimadores finalizada. Accuracy: 1.0000

¡Tracking finalizado! Puedes abrir la consola de MLflow en tu terminal ejecutando: mlflow ui


## 10. Referencias y Recursos

- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [scikit-learn: Model Persistence](https://scikit-learn.org/stable/model_persistence.html)
- [Docker Getting Started](https://docs.docker.com/get-started/)
- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)

---

📎 **Notebook anterior:** [12. CPU, GPU y Metal](./12_cpu_gpu_metal.ipynb)  
📎 **Este es el último notebook del curso.** ¡Felicidades por completar el recorrido! 🎉